# 🧠 DaMi's brain: Ollama on a free Kaggle GPU

This notebook runs an open LLM with [Ollama](https://docs.ollama.com/) on Kaggle's free GPU, and makes it
reachable from your laptop through an [ngrok](https://ngrok.com/docs/what-is-ngrok) tunnel.

* **Start it from your terminal** with `dami kaggle start` (recommended), or here with *Run All*.
* It needs the Kaggle secret `NGROK_AUTHTOKEN` (*Add-ons → Secrets*), GPU **T4 x2** and internet **on**.
* It **stops by itself** after `IDLE_MINUTES` without requests, to save your weekly GPU quota.

In [ ]:
# PARAMETERS (`dami kaggle start` changes these values before it uploads the notebook)
MODEL = 'qwen3.5:9b'
IDLE_MINUTES = 20
MAX_HOURS = 4
NGROK_DOMAIN = ''
NGROK_AUTHTOKEN_FALLBACK = ''

## 1. Check the GPU, install Ollama and pyngrok

In [ ]:
import json
import os
import subprocess
import time
import urllib.request

subprocess.run(["nvidia-smi"], check=False)
subprocess.run("sudo apt-get install -y -qq zstd", shell=True, check=True)  # the Ollama installer needs zstd
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.run(["pip", "install", "-q", "pyngrok>=9,<10"], check=True)

## 2. Start the Ollama server

In [ ]:
OLLAMA_PORT = 11434
server_env = dict(
    os.environ,
    OLLAMA_HOST=f"127.0.0.1:{OLLAMA_PORT}",  # only local; the outside world comes in through ngrok
    OLLAMA_KEEP_ALIVE="30m",                 # keep the model in GPU memory between requests
    OLLAMA_CONTEXT_LENGTH="8192",            # smaller context = less GPU memory
)
ollama_log = open("ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=server_env, stdout=ollama_log, stderr=subprocess.STDOUT)

def ollama_api(path, payload=None, timeout=10):
    data = json.dumps(payload).encode() if payload is not None else None
    req = urllib.request.Request(f"http://127.0.0.1:{OLLAMA_PORT}{path}", data=data,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read() or b"{}")

for _ in range(60):
    try:
        print("Ollama", ollama_api("/api/version")["version"], "is running.")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. See ollama.log")

## 3. Download the model and warm it up

In [ ]:
subprocess.run(["ollama", "pull", MODEL], check=True)
t = time.time()
reply = ollama_api("/api/generate", {"model": MODEL, "prompt": "Say hello to the DaMi detectives in 5 words.",
                                     "stream": False, "think": False}, timeout=600)
print(f"{reply.get('response', '').strip()}   ({time.time() - t:.1f}s, first load included)")
subprocess.run(["ollama", "ps"], check=False)

## 4. Open the ngrok tunnel

In [ ]:
from pyngrok import conf, ngrok

# `dami kaggle start` writes the token into the PARAMETERS cell (Kaggle secrets do not reach
# versions pushed from the API). When you run the notebook by hand, the Kaggle secret is used.
token = NGROK_AUTHTOKEN_FALLBACK
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    except Exception as e:
        print(f"Could not read the Kaggle secret NGROK_AUTHTOKEN ({e}).")
if not token:
    raise RuntimeError("No ngrok token. Use `dami kaggle start` with NGROK_AUTHTOKEN in .env, "
                       "or add the NGROK_AUTHTOKEN secret to this notebook (Add-ons -> Secrets).")

# pyngrok 9 uses ngrok's new "endpoints" API by default, which has no host_header option.
# Ollama only answers requests whose Host header is localhost, so we use the classic
# "tunnels" API (config version 2), which can rewrite the Host header.
ngrok_config = conf.PyngrokConfig(config_version="2", config_path=os.path.abspath("ngrok-dami.yml"), auth_token=token)
ngrok.kill()  # stop an ngrok agent left over from an earlier run of this cell
options = {"host_header": f"localhost:{OLLAMA_PORT}"}
if NGROK_DOMAIN:
    options["domain"] = NGROK_DOMAIN
tunnel = ngrok.connect(OLLAMA_PORT, pyngrok_config=ngrok_config, **options)
print("=" * 70)
print("DaMi's tunnel URL (`dami kaggle start` writes it into .env; when you run the notebook by hand, copy it):")
print()
print(f"    OLLAMA_HOST={tunnel.public_url}")
print("=" * 70)

## 5. Keep running until nobody uses the brain

This cell watches the tunnel. It stops the notebook after `IDLE_MINUTES` without requests,
or after `MAX_HOURS`. If you run the notebook by hand, you can also stop this cell.

In [ ]:
def tunnel_requests():
    try:
        with urllib.request.urlopen("http://127.0.0.1:4040/api/tunnels", timeout=5) as r:
            tunnels = json.loads(r.read())["tunnels"]
        return sum(t.get("metrics", {}).get("http", {}).get("count", 0) for t in tunnels)
    except Exception:
        return None

started = last_activity = time.time()
last_count = tunnel_requests()
try:
    while True:
        time.sleep(30)
        count = tunnel_requests()
        if count is not None and count != last_count:
            last_count, last_activity = count, time.time()
        idle = (time.time() - last_activity) / 60
        if int(time.time() - started) % 300 < 30:
            print(f"[{time.strftime('%H:%M')}] requests so far: {last_count}, idle for {idle:.0f} min")
        if idle >= IDLE_MINUTES:
            print(f"No requests for {IDLE_MINUTES} minutes: stopping to save GPU quota.")
            break
        if time.time() - started >= MAX_HOURS * 3600:
            print(f"Reached MAX_HOURS={MAX_HOURS}: stopping.")
            break
except KeyboardInterrupt:
    print("Stopped by hand.")
finally:
    ngrok.kill(pyngrok_config=ngrok_config)
    ollama_proc.terminate()
    print("Tunnel closed and Ollama stopped. Bye!")